In [0]:
# Databricks notebook source
import pyspark.sql.functions as F
from pyspark.sql.window import Window

storage_account = "vesseldatavd99"
storage_key = dbutils.secrets.get(scope="vessel-secrets", key="storage-key")
  # paste at runtime only, never commit


In [0]:

spark.conf.set(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)
spark.conf.set("spark.sql.shuffle.partitions", "64")

BRONZE = f"abfss://bronze@{storage_account}.dfs.core.windows.net/ais_bronze_v2"
SILVER = f"abfss://silver@{storage_account}.dfs.core.windows.net/ais_clean"

ORIG_COLS = ["MMSI", "BaseDateTime", "LAT", "LON", "SOG", "COG", "Heading",
             "VesselName", "IMO", "CallSign", "VesselType", "Status", "Length",
             "Width", "Draft", "Cargo", "TransceiverClass"]

bronze = spark.read.format("delta").load(BRONZE)

In [0]:
from delta.tables import DeltaTable

def transform(src):
    deduped = src.dropDuplicates(ORIG_COLS)
    w = Window.partitionBy("MMSI", "BaseDateTime")
    deduped = deduped.withColumn("key_conflict", F.count("*").over(w) > 1)

    typed = (deduped
        .withColumn("ts",  F.to_timestamp("BaseDateTime"))
        .withColumn("lat", F.col("LAT").cast("double"))
        .withColumn("lon", F.col("LON").cast("double"))
        .withColumn("sog", F.col("SOG").cast("double"))
        .withColumn("cog", F.col("COG").cast("double"))
        .withColumn("hdg", F.col("Heading").cast("double")))

    typed = typed.withColumn("cast_failed",
        (F.col("ts").isNull()  & F.col("BaseDateTime").isNotNull()) |
        (F.col("lat").isNull() & F.col("LAT").isNotNull()) |
        (F.col("lon").isNull() & F.col("LON").isNotNull()) |
        (F.col("sog").isNull() & F.col("SOG").isNotNull()) |
        (F.col("cog").isNull() & F.col("COG").isNotNull()) |
        (F.col("hdg").isNull() & F.col("Heading").isNotNull()))

    typed = (typed
        .withColumn("sog", F.when(F.col("sog") == 102.3, None).otherwise(F.col("sog")))
        .withColumn("hdg", F.when(F.col("hdg") == 511, None).otherwise(F.col("hdg"))))

    return (typed
        .withColumn("validation_status",
            F.when(F.col("cast_failed") | F.col("ts").isNull()
                   | F.col("lat").isNull() | F.col("lon").isNull()
                   | (F.col("lat") < -90) | (F.col("lat") > 90)
                   | (F.col("lon") < -180) | (F.col("lon") > 180), "INVALID")
             .when((F.col("sog") > 50) | F.col("key_conflict"), "FLAGGED")
             .otherwise("VALID"))
        .withColumn("processed_timestamp", F.current_timestamp())
        .select("MMSI", "ts", "lat", "lon", "sog", "cog", "hdg",
                "VesselName", "IMO", "CallSign", "VesselType", "Status",
                "Length", "Width", "Draft", "Cargo", "TransceiverClass",
                "key_conflict", "validation_status",
                "source_file_date", "ingestion_id", "ingestion_timestamp",
                "processed_timestamp"))

bronze = spark.read.format("delta").load(BRONZE)
from delta.tables import DeltaTable

# Add a date here only when Bronze has changed for that day and Silver needs a rebuild
REBUILD_DAYS = []

bronze = spark.read.format("delta").load(BRONZE)
bronze_days = {r[0] for r in bronze.select("source_file_date").distinct().collect()}

table_exists = DeltaTable.isDeltaTable(spark, SILVER)
if table_exists:
    silver_days = {r[0] for r in
                   spark.read.format("delta").load(SILVER)
                   .select("source_file_date").distinct().collect()}
else:
    silver_days = set()

todo = sorted(d for d in bronze_days if d not in silver_days or d in REBUILD_DAYS)
print("days to write:", todo or "none")

for d in todo:
    out = transform(bronze.filter(F.col("source_file_date") == d))
    writer = out.write.format("delta").partitionBy("source_file_date").mode("overwrite")
    if table_exists:
        # Replaces only this day's partition. Works for new days too.
        writer = writer.option("replaceWhere", f"source_file_date = '{d.isoformat()}'")
    writer.save(SILVER)
    table_exists = True
    spark.catalog.clearCache()
    print("silver written:", d, flush=True)

print("done")

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:136)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecutio

In [0]:
silver = spark.read.format("delta").load(SILVER)

total = silver.count()
print(f"Silver total: {total:,}")

display(silver.groupBy("source_file_date", "validation_status")
        .count()
        .orderBy("source_file_date", "validation_status"))

display(silver.groupBy("validation_status")
        .agg(F.count("*").alias("rows"),
             F.sum(F.col("key_conflict").cast("int")).alias("key_conflict_rows")))

Silver total: 110,608,180


source_file_date,validation_status,count
2019-01-01,FLAGGED,72
2019-01-01,VALID,7515821
2019-01-02,FLAGGED,130
2019-01-02,VALID,7168080
2019-01-03,FLAGGED,132
2019-01-03,VALID,6722352
2019-01-04,FLAGGED,105
2019-01-04,VALID,6949337
2019-01-05,FLAGGED,246
2019-01-05,VALID,6849316


validation_status,rows,key_conflict_rows
VALID,110605339,0
FLAGGED,2841,570


com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
silver.printSchema()

root
 |-- MMSI: string (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- sog: double (nullable = true)
 |-- cog: double (nullable = true)
 |-- hdg: double (nullable = true)
 |-- VesselName: string (nullable = true)
 |-- IMO: string (nullable = true)
 |-- CallSign: string (nullable = true)
 |-- VesselType: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Length: string (nullable = true)
 |-- Width: string (nullable = true)
 |-- Draft: string (nullable = true)
 |-- Cargo: string (nullable = true)
 |-- TransceiverClass: string (nullable = true)
 |-- key_conflict: boolean (nullable = true)
 |-- validation_status: string (nullable = true)
 |-- source_file_date: date (nullable = true)
 |-- ingestion_id: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- processed_timestamp: timestamp (nullable = true)



com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
silver.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in silver.columns]).show(truncate=False)

+----+---+---+---+------+---+--------+----------+--------+--------+----------+--------+--------+--------+--------+--------+----------------+------------+-----------------+----------------+------------+-------------------+-------------------+
|MMSI|ts |lat|lon|sog   |cog|hdg     |VesselName|IMO     |CallSign|VesselType|Status  |Length  |Width   |Draft   |Cargo   |TransceiverClass|key_conflict|validation_status|source_file_date|ingestion_id|ingestion_timestamp|processed_timestamp|
+----+---+---+---+------+---+--------+----------+--------+--------+----------+--------+--------+--------+--------+--------+----------------+------------+-----------------+----------------+------------+-------------------+-------------------+
|0   |0  |0  |0  |133601|0  |62434401|5604708   |62582973|14967338|7054372   |22343331|14482048|29953834|81263809|73982771|0               |0           |0                |0               |0           |0                  |0                  |
+----+---+---+---+------+---+---

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
(silver.groupBy("VesselType")
    .agg(F.count("*").alias("rows"),
         F.avg("sog").alias("avg_sog"),
         F.max("sog").alias("max_sog"))
    .orderBy(F.desc("rows"))
    .show(20))

+----------+--------+-------------------+-------+
|VesselType|    rows|            avg_sog|max_sog|
+----------+--------+-------------------+-------+
|        31|45466854| 1.6157681161851716|  102.2|
|        37|17060341| 0.5029062223891417|  102.2|
|        60| 8513882| 3.5469059106301106|  102.2|
|        90| 8462113| 1.2911787673842645|  102.2|
|        30| 8189147|    1.4665074703138|  102.2|
|        70| 7823526|  5.236615296946442|  102.2|
|      NULL| 7054372|    2.4090393605961|  102.2|
|        80| 3642335|  5.090578428491839|   64.3|
|        36| 1886135| 0.8293993642920748|   49.3|
|        52|  575086|  1.081106182302524|   16.6|
|         0|  434875| 0.9406875110662568|  102.2|
|        50|  283370|   2.68081412993627|   41.1|
|        57|  148637|  1.445794788646093|   38.3|
|        33|  125754| 1.4481229076045792|   58.9|
|        51|   79119| 2.7049811044122807|   45.9|
|        35|   73160|  1.498742482230734|   47.5|
|        32|   67775|  1.510931920795598|   12.5|


com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
spark.sql(f"DESCRIBE HISTORY delta.`{SILVER}`").select("version", "timestamp", "operation").show(truncate=False)

old = spark.read.format("delta").option("versionAsOf", 0).load(SILVER)
print("version 0 rows:", old.count())

+-------+-------------------+------------+
|version|timestamp          |operation   |
+-------+-------------------+------------+
|44     |2026-10-26 06:42:06|WRITE       |
|43     |2026-10-26 06:39:48|WRITE       |
|42     |2026-10-26 06:37:21|WRITE       |
|41     |2026-10-26 06:34:58|WRITE       |
|40     |2026-10-26 06:32:43|WRITE       |
|39     |2026-10-26 06:30:13|WRITE       |
|38     |2026-10-26 06:27:46|WRITE       |
|37     |2026-10-26 06:25:13|WRITE       |
|36     |2026-10-26 06:22:32|WRITE       |
|35     |2026-10-26 06:20:08|WRITE       |
|34     |2026-10-26 06:17:44|WRITE       |
|33     |2026-10-26 06:15:15|WRITE       |
|32     |2026-10-26 06:12:47|WRITE       |
|31     |2026-10-26 06:10:10|WRITE       |
|30     |2026-10-26 06:07:39|WRITE       |
|29     |2026-10-26 06:05:10|WRITE       |
|28     |2026-10-26 05:52:20|WRITE       |
|27     |2026-10-26 05:49:46|WRITE       |
|26     |2026-10-26 00:37:20|VACUUM END  |
|25     |2026-10-26 00:37:12|VACUUM START|
+-------+--

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
spark.sql(f"ALTER TABLE delta.`{SILVER}` SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print("CDF enabled")

CDF enabled


com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
before = spark.sql(f"DESCRIBE DETAIL delta.`{SILVER}`").select("numFiles", "sizeInBytes").collect()[0]
print("before: files =", before["numFiles"], " bytes =", before["sizeInBytes"])

before: files = 174  bytes = 1754429643


com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
import time
t0 = time.time()
n = spark.read.format("delta").load(SILVER).filter("MMSI = '209890000'").count()
print("rows:", n, " query seconds:", round(time.time() - t0, 2))

rows: 13898  query seconds: 8.05


com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
spark.sql(f"OPTIMIZE delta.`{SILVER}` ZORDER BY (MMSI)").show(truncate=False)

+------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|path                                                        |metrics                                                                                                                                                                                                                                                                                                                     |
+------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
after = spark.sql(f"DESCRIBE DETAIL delta.`{SILVER}`").select("numFiles", "sizeInBytes").collect()[0]
print("after: files =", after["numFiles"], " bytes =", after["sizeInBytes"])

after: files = 16  bytes = 1529530010


com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
(spark.sql(f"DESCRIBE HISTORY delta.`{SILVER}`")
    .filter("operation = 'OPTIMIZE'")
    .select("version", "timestamp", "operationParameters", "operationMetrics")
    .show(truncate=False))

+-------+-------------------+--------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operationParameters                                                 |operationMetrics                                                                                                                                                                                                                                                       |
+-------+-------------------+--------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
(silver.filter("MMSI = '209890000'")
    .select("ts", "lat", "lon", "sog", "cog", "validation_status")
    .orderBy("ts")
    .show(20, truncate=False))

+-------------------+--------+----------+---+-----+-----------------+
|ts                 |lat     |lon       |sog|cog  |validation_status|
+-------------------+--------+----------+---+-----+-----------------+
|2019-01-01 00:01:27|33.75648|-118.21828|0.0|183.0|VALID            |
|2019-01-01 00:04:26|33.75652|-118.21827|0.0|183.0|VALID            |
|2019-01-01 00:07:28|33.75653|-118.21827|0.0|183.0|VALID            |
|2019-01-01 00:10:26|33.7565 |-118.21827|0.0|183.0|VALID            |
|2019-01-01 00:13:26|33.75652|-118.21828|0.0|183.0|VALID            |
|2019-01-01 00:16:28|33.75652|-118.21828|0.0|183.0|VALID            |
|2019-01-01 00:19:26|33.75652|-118.21828|0.0|183.0|VALID            |
|2019-01-01 00:22:26|33.75652|-118.21827|0.0|183.0|VALID            |
|2019-01-01 00:25:29|33.7565 |-118.21828|0.0|183.0|VALID            |
|2019-01-01 00:28:26|33.7565 |-118.21828|0.0|183.0|VALID            |
|2019-01-01 00:31:26|33.7565 |-118.21827|0.0|183.0|VALID            |
|2019-01-01 00:34:29

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage

In [0]:
spark.sql(f"VACUUM delta.`{SILVER}` RETAIN 168 HOURS")
print("vacuum done")

vacuum done


com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:138)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:730)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:448)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:448)
	at com.databricks.spark.chauffeur.ChauffeurState.cancelExecution(ChauffeurState.scala:1315)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:1032)
	at com.databricks.logging.Usage